In [1]:
using Revise
using PrecisionCarriers
using Random
using Pkg
Pkg.develop(url="/home/antonr/repos/QEDbase.jl")
Pkg.develop(url="/home/antonr/repos/QEDcore.jl")
Pkg.develop(url="/home/antonr/repos/QEDprocesses.jl")
Pkg.develop(url="/home/antonr/repos/QEDFeynmanDiagrams.jl")
using QEDbase, QEDcore, QEDprocesses, QEDFeynmanDiagrams

   Resolving package versions...
  No Changes to `~/repos/PrecisionCarriers.jl/Project.toml`
  No Changes to `~/repos/PrecisionCarriers.jl/Manifest.toml`
   Resolving package versions...
  No Changes to `~/repos/PrecisionCarriers.jl/Project.toml`
  No Changes to `~/repos/PrecisionCarriers.jl/Manifest.toml`
   Resolving package versions...
  No Changes to `~/repos/PrecisionCarriers.jl/Project.toml`
  No Changes to `~/repos/PrecisionCarriers.jl/Manifest.toml`
   Resolving package versions...
  No Changes to `~/repos/PrecisionCarriers.jl/Project.toml`
  No Changes to `~/repos/PrecisionCarriers.jl/Manifest.toml`
┌ Warning: Circular dependency detected.
│ Precompilation will be skipped for dependencies in this cycle:
│  ┌ QEDprocesses
│  └─ PrecisionCarriers
│ Precompilation will also be skipped for the following, which depend on the above cycle:
│   PrecisionCarriers → RandomExt
└ @ Base.Precompilation precompilation.jl:651


In [2]:
# process setup
RNG = MersenneTwister(0)

FLOAT_T = PrecisionCarrier{Float64}

N = 3
PROC = ScatteringProcess(
    (Electron(), Photon()),
    (Electron(), ntuple(_ -> Photon(), N)...,),
)
MODEL = PerturbativeQED()
PSL = FlatPhaseSpaceLayout(ComptonRestSystem())

FlatPhaseSpaceLayout{TwoBodyTargetSystem{Energy{2}}}(TwoBodyTargetSystem{Energy{2}}(Energy{2}()))

In [3]:
# input generation
@show IN_N = phase_space_dimension(PROC, MODEL, in_phase_space_layout(PSL))
@show OUT_N = phase_space_dimension(PROC, MODEL, PSL)

OMEGA = one(FLOAT_T)

IN_COORDS = [(OMEGA,)]
OUT_COORDS = [ntuple(_ -> rand(RNG, FLOAT_T), OUT_N) for _ in 1:10000];

IN_N = phase_space_dimension(PROC, MODEL, in_phase_space_layout(PSL)) = 1
OUT_N = phase_space_dimension(PROC, MODEL, PSL) = 16


In [4]:
out_matrix = fill(zero(FLOAT_T), (length(OUT_COORDS)))

moms = QEDbase._build_momenta.(PROC, MODEL, PSL, IN_COORDS, OUT_COORDS)

10000-element Vector{Tuple{Tuple{SFourMomentum{PrecisionCarrier{Float64}}, SFourMomentum{PrecisionCarrier{Float64}}}, NTuple{4, SFourMomentum{PrecisionCarrier{Float64}}}}}:
 (([1.0 <ε=0>, 0.0 <ε=0>, 0.0 <ε=0>, 0.0 <ε=0>], [1.0 <ε=0>, 0.0 <ε=0>, 0.0 <ε=0>, 1.0 <ε=0>]), ([1.020121883436576 <ε=1>, 0.12282167805320106 <ε=0>, 0.07155469271712732 <ε=1>, 0.14298048264200286 <ε=5>], [0.5311166986829562 <ε=1>, 0.12304264713919089 <ε=2>, -0.14228677580138452 <ε=1>, 0.49668896508469884 <ε=1>], [0.15806906229858628 <ε=1>, -0.08394694629976823 <ε=2>, -0.011714560173030443 <ε=3>, 0.1334222910268394 <ε=0>], [0.2906923555818818 <ε=2>, -0.16191737889262367 <ε=2>, 0.08244664325728766 <ε=2>, 0.22690826124645908 <ε=0>]))
 (([1.0 <ε=0>, 0.0 <ε=0>, 0.0 <ε=0>, 0.0 <ε=0>], [1.0 <ε=0>, 0.0 <ε=0>, 0.0 <ε=0>, 1.0 <ε=0>]), ([1.2478463641740114 <ε=1>, -0.2291164587661294 <ε=4>, -0.4039731014696367 <ε=3>, 0.5843217693991647 <ε=1>], [0.02892338380069347 <ε=1>, 0.0031007977293180705 <ε=18>, 0.020543168084273686 <ε=1>

In [ ]:
using ProgressMeter
using Base.Threads

@showprogress for i in eachindex(out_matrix)
    out_matrix[i] = unsafe_differential_cross_section(
        PhaseSpacePoint(
            PROC,
            MODEL,
            PSL,
            moms[i][1],
            moms[i][2]
        )
    )
end

out_matrix

In [ ]:
using CairoMakie

f = Figure()
ax = Axis(f[1, 1]; title = "epsilons ($PROC, $(eltype(FLOAT_T)))", xlabel = "epsilons", ylabel = "number", xscale=log10)

@show m = maximum(PrecisionCarriers._no_epsilons.(out_matrix))

sig_digits = PrecisionCarriers._no_epsilons.(out_matrix)
co_sig_digits = hist!(ax, sig_digits; bins = m)

f

In [ ]:
save("$(N)_compton_epsilons_$(eltype(FLOAT_T)).pdf", f)